In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

SELECT_PRODUCT_MAPPER = {
    '1':'TIERS_SIMPLE',
    '4': 'SUR_MESURE',
    '3': 'TOUS_RISQUE',
    '1000':'MOTO'
}

TARGET_COLUMNS = [
    'agent.email',
    'agent.userName',
    'amount',
    'carRegNo',
    'contractDuration',
    'contractDuration_month',
    'createdAt',
    'effectDate',
    'email','endDate','isRenewal', 
    'fullName',
    'operator',
    'phoneNo',
    'policyNo',
    'quote.agentCode',
    'quote.brand',
    'quote.carActualValue',
    'quote.carNewValue',
    'quote.carParkingLocation',
    'quote.category',
    'quote.energy',
    'quote.firstDriveDate',
    'quote.model',
    'quote.noOfSeat',
    'quote.power',
    'quote.quoteValues.detailsPrime.accessoires',
    'quote.quoteValues.detailsPrime.cedeao',
    'quote.quoteValues.detailsPrime.fga',
    'quote.quoteValues.detailsPrime.primeNette',
    'quote.quoteValues.detailsPrime.taxes',
    'quote.quoteValues.listeGaranties.garantieDefenseRecours',
    'quote.quoteValues.listeGaranties.garantieRC',
    'quote.quoteValues.listeGaranties.garantieVie',
    'quote.vehicleType',
    'quote.vtc', 
    'status', 
    'selectedProduct',
    'selectedProduct_map'
    ]

BASE_FILE_PATH = "data/Base_Leadway.xlsx"
MONTH_FILTE_PATH = "data/EXTRACTION.xlsx"

## BASE FILE PROCESSING

In [2]:
df_base = pd.read_excel(BASE_FILE_PATH, dtype='object')

In [2]:
df_base = pd.read_excel(BASE_FILE_PATH, dtype='object')

In [3]:
import pandas as pd
from datetime import datetime

# ============================================================
# 🎯 Objectif :
# Extraire les contrats arrivant à échéance en novembre 2025,
# créer le lien de renouvellement, puis générer un fichier Excel.
# ============================================================

# --- 🔹 0. Chargement de la base source ---
# (Remplace 'df_base' par ton DataFrame réel ou un import CSV)
df = df_base.copy()

# --- 🔹 1. Définition des constantes ---
BASE_URL = "https://ci.leadway.com/auto/quote/"
PROCESSING_COLUMNS = [
    "NOM COMPLET",
    "PRODUIT",
    "PRIME TTC",
    "IMMATRICULATION",
    "TELEPHONE",
    "DUREE ( MOIS)",
    "DATE ECHEANCE",
    "LIEN DE RENOUVELLEMENT",
    "EMAIL AGENT",
]

# --- 🔹 2. Conversion du format de date ---
df["DATE ECHEANCE"] = pd.to_datetime(df["DATE ECHEANCE"], errors="coerce")

# --- 🔹 3. Définir la période d’analyse (Octobre 2025) ---
debut_jan = datetime(2026, 1, 1)
fin_jan = datetime(2026, 1, 31)

# --- 🔹 4. Filtrage des contrats à échéance ---
df_janvier = df[
    (df["DATE ECHEANCE"] >= debut_jan) &
    (df["DATE ECHEANCE"] <= fin_jan)
].copy()

# --- 🔹 5. Génération du lien de renouvellement ---
df_janvier["LIEN DE RENOUVELLEMENT"] = df_janvier["NUMERO DE COTATION"].apply(
    lambda x: f"{BASE_URL}{str(x).strip()}" if pd.notna(x) and str(x).strip() else ""
)

# --- 🔹 6. Sélectionner uniquement les colonnes pertinentes ---
colonnes_finales = [col for col in PROCESSING_COLUMNS if col in df_janvier.columns]
data_to_generate = df_janvier[colonnes_finales].copy()

# --- 🔹 7. Nettoyage des données (facultatif mais utile) ---
data_to_generate = data_to_generate.drop_duplicates().reset_index(drop=True)

# --- 🔹 8. Vérification et aperçu ---
print(f"✅ DataFrame 'data_to_generate' créé avec {len(data_to_generate)} lignes et {len(data_to_generate.columns)} colonnes.")
print("🔹 Aperçu :")
print(data_to_generate.head())

# --- 🔹 9. Export Excel ---
output_file = "Contrats_Echeance_Janvier2025.xlsx"
try:
    data_to_generate.to_excel(output_file, index=False)
    print(f"📁 Fichier exporté avec succès : {output_file}")
except Exception as e:
    print(f"⚠️ Erreur lors de l’export du fichier : {e}")


✅ DataFrame 'data_to_generate' créé avec 29276 lignes et 9 colonnes.
🔹 Aperçu :
                    NOM COMPLET PRODUIT PRIME TTC IMMATRICULATION   TELEPHONE  \
0               OUEDRAOGO ADAMA    MOTO      8580        9386JJ04  0787608725   
1    KOUAKOU ATTOUNGBRE JOACHIM    MOTO      8580        6710LK04  0708809171   
2  COULIBALY AICHA TIELOUROUGO     MOTO      6000   CHCBC3K804307  0701799995   
3                OUATTARA SIAKA    MOTO      6000        3910KA01  0701799995   
4                  KABORE DAVID    MOTO      6000        7564KE03  0594158026   

  DUREE ( MOIS) DATE ECHEANCE  \
0            12    2026-01-10   
1            12    2026-01-26   
2            12    2026-01-17   
3            12    2026-01-10   
4            12    2026-01-03   

                             LIEN DE RENOUVELLEMENT               EMAIL AGENT  
0  https://ci.leadway.com/auto/quote/SN437714342024     celestaka82@gmail.com  
1  https://ci.leadway.com/auto/quote/SNC5A350262024  nouradiallo224@gmail.

In [4]:
import os
import pandas as pd

# --- Création du dossier /data (au même niveau que ton script actuel) ---
base_dir = os.getcwd()  # récupère le dossier courant
output_dir = os.path.join(base_dir, "data")
os.makedirs(output_dir, exist_ok=True)

# --- Enregistre le fichier data_to_generate.xlsx dans ce dossier ---
output_path = os.path.join(output_dir, "data_to_generate.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    data_to_generate.to_excel(writer, sheet_name="data_to_generate", index=False)

print(f"📁 Fichier correctement enregistré : {output_path}")


📁 Fichier correctement enregistré : /Users/y-kouadio/Documents/WORKSPACE/agent_report/data/data_to_generate.xlsx
